In [20]:
import pandas as pd

In [21]:

# ---------------------------------------------------------
# 1. CONFIGURACIÓN DE LA JUGADA GANADORA
# ---------------------------------------------------------
# Reemplaza estos valores con los resultados del sorteo real
numeros_ganadores = [1, 28, 31, 37, 11, 9]  # Los 6 números principales
bolilla_yapa = 26                             # La bolilla Yapa
bolillas_siosi = [8, 50]                 # Array de bolillas del "Sí o Sí"

# Nombre de tu archivo excel
archivo_excel = 'df_possible_v4_18012026.xlsx'

In [22]:
# ---------------------------------------------------------
# 2. CARGA DE DATOS
# ---------------------------------------------------------
# Leemos el excel. Asumimos que las columnas se llaman b1, b2, b3, b4, b5, b6
try:
    df = pd.read_excel(archivo_excel)
    print(f"Cargadas {len(df)} jugadas exitosamente.")
except FileNotFoundError:
    print("Error: No se encontró el archivo. Verifica el nombre.")
    # Creamos data de prueba si no hay archivo para que veas que el código funciona
    data_fake = {
        'b1': [2, 10, 15, 22, 30, 41], # Ganador Jackpot
        'b2': [2, 10, 15, 22, 30, 4],  # 5 aciertos + Yapa
        'b3': [2, 10, 15, 22, 30, 5],  # 5 aciertos + Si o Si
        'b4': [1, 2, 3, 4, 5, 6],      # Nada
        'b5': [2, 10, 15, 8, 9, 41],   # 4 aciertos
        'b6': [1, 1, 1, 1, 1, 1]       # Relleno (b6 real estaría en su columna)
    }
    # Ajuste para estructura de ejemplo: Pandas espera columnas, no filas en el dict
    # pero para el ejemplo visual sirve entender la lógica
    pass

Cargadas 2118 jugadas exitosamente.


In [23]:
# ---------------------------------------------------------
# 3. LÓGICA DE PREMIOS (Función Núcleo)
# ---------------------------------------------------------
def verificar_premio(fila):
    # Convertimos la jugada de la fila a un conjunto (set) para comparar rápido
    mi_jugada = {fila['b1'], fila['b2'], fila['b3'], fila['b4'], fila['b5'], fila['b6']}
    
    # Conjuntos de ganadores
    set_ganador = set(numeros_ganadores)
    set_siosi = set(bolillas_siosi)
    
    # Calculamos aciertos principales (intersección)
    aciertos = len(mi_jugada.intersection(set_ganador))
    
    # Verificamos si tenemos la Yapa (si la yapa está dentro de mis 6 números)
    tiene_yapa = bolilla_yapa in mi_jugada
    
    # Verificamos si tenemos alguna del Si o Si
    # (Intersección mayor a 0 significa que al menos una coincide)
    tiene_siosi = len(mi_jugada.intersection(set_siosi)) > 0
    
    # --- ÁRBOL DE DECISIÓN DE PREMIOS ---
    
    # 6 Aciertos
    if aciertos == 6:
        return "JACKPOT", "Pozo Millonario"
    
    # 5 Aciertos
    if aciertos == 5:
        if tiene_yapa:
            return "5 Aciertos + Yapa", 50000
        elif tiene_siosi:
            return "Sí o Sí", "Premio Si o Si" # Valor variable, usualmente se reparte
        else:
            return "5 Aciertos", 5000
            
    # 4 Aciertos
    if aciertos == 4:
        if tiene_yapa:
            return "4 Aciertos + Yapa", 500
        else:
            return "4 Aciertos", 100
            
    # 3 Aciertos
    if aciertos == 3:
        if tiene_yapa:
            return "3 Aciertos + Yapa", 50
        else:
            return "3 Aciertos", 10
            
    # 2 Aciertos
    if aciertos == 2:
        if tiene_yapa:
            return "2 Aciertos + Yapa", 5
            
    return "Sin Premio", 0

In [24]:
# ---------------------------------------------------------
# 4. EJECUCIÓN
# ---------------------------------------------------------

# Aplicamos la función a cada fila del Excel
# Esto devuelve dos columnas nuevas: 'Tipo_Premio' y 'Monto'
resultados = df.apply(verificar_premio, axis=1, result_type='expand')
df['Tipo_Premio'] = resultados[0]
df['Monto'] = resultados[1]

In [25]:
# ---------------------------------------------------------
# 5. REPORTE DE RESULTADOS
# ---------------------------------------------------------
print("\n--- RESUMEN DE RESULTADOS ---")
conteo_premios = df['Tipo_Premio'].value_counts()
print(conteo_premios)

print("\n--- DETALLE DE PREMIOS GANADOS ---")
# Filtramos solo los que ganaron algo (excluyendo "Sin Premio")
ganadores = df[df['Tipo_Premio'] != "Sin Premio"]

if not ganadores.empty:
    print(ganadores[['b1', 'b2', 'b3', 'b4', 'b5', 'b6', 'Tipo_Premio', 'Monto']])
    
    # Cálculo de dinero total (excluyendo pozos variables como Jackpot o Si o Si)
    monto_fijo_total = pd.to_numeric(ganadores['Monto'], errors='coerce').sum()
    print(f"\nTotal ganado en premios fijos: S/ {monto_fijo_total:,.2f}")
else:
    print("Suerte para la próxima, no hubo premios en este lote.")

# Opcional: Guardar los ganadores en un nuevo Excel
ganadores.to_excel(f"{archivo_excel}.REVIEW.xlsx", index=False)


--- RESUMEN DE RESULTADOS ---
Tipo_Premio
Sin Premio           2025
3 Aciertos             61
2 Aciertos + Yapa      32
Name: count, dtype: int64

--- DETALLE DE PREMIOS GANADOS ---
      b1  b2  b3  b4  b5  b6 Tipo_Premio  Monto
59     1   9  14  18  28  35  3 Aciertos     10
62     1   9  14  23  28  38  3 Aciertos     10
64     1   9  14  24  28  35  3 Aciertos     10
65     1   9  14  24  28  38  3 Aciertos     10
70     1   9  14  25  37  44  3 Aciertos     10
...   ..  ..  ..  ..  ..  ..         ...    ...
2077   9  24  31  37  41  48  3 Aciertos     10
2078   9  24  31  37  42  46  3 Aciertos     10
2079   9  24  31  37  42  48  3 Aciertos     10
2080   9  24  31  37  42  49  3 Aciertos     10
2081   9  24  31  37  42  50  3 Aciertos     10

[93 rows x 8 columns]

Total ganado en premios fijos: S/ 770.00
